
- **Lớp đầu vào (Input Layer)**: 784 nơ-ron (tương ứng ảnh 28x28 pixel). Dữ liệu pixel cần được chuẩn hóa về từ 0.0 (trắng) đến 1.0 (đen).
- **Lớp ẩn (Hidden Layer)**: $n$ nơ-ron. Chúng ta sẽ thử nghiệm thay đổi $n$ (ví dụ: $n=15, n=30, n=100$) để xem kết quả thay đổi ra sao.
- **Lớp đầu ra (Output Layer)**: 10 nơ-ron (đại diện cho các số 0-9). Cơ chế là "Winner-takes-all": Nơ-ron nào có giá trị cao nhất thì dự đoán là số đó.

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# --- 1. Class NeuralNetwork ---
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate

        # Khởi tạo trọng số ngẫu nhiên
        np.random.seed(42)
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    # Hàm kích hoạt Sigmoid (theo yêu cầu đề bài mô tả về kích hoạt 0-1)
    # Tuy nhiên ReLU thường tốt hơn, bạn có thể thử đổi sang ReLU nếu muốn
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    # Hàm Softmax cho đầu ra
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    # Forward Propagation
    def forward(self, X):
        # Input -> Hidden
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1) # Dùng Sigmoid ở lớp ẩn

        # Hidden -> Output
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.softmax(self.Z2)
        return self.A2

    # Backpropagation
    def backward(self, X, y, output):
        m = X.shape[0]

        # Tính đạo hàm tại Output
        dZ2 = output - y
        dW2 = (1/m) * np.dot(self.A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

        # Tính đạo hàm tại Hidden
        dZ1 = np.dot(dZ2, self.W2.T) * self.sigmoid_derivative(self.Z1)
        dW1 = (1/m) * np.dot(X.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        # Cập nhật trọng số
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

    # Huấn luyện
    def train(self, X, y, epochs):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)

            if epoch % 50 == 0:
                # Tính loss đơn giản để theo dõi
                loss = -np.mean(y * np.log(output + 1e-9))
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # Dự đoán (Chọn nơ-ron có giá trị cao nhất)
    def predict(self, X):
        output = self.forward(X)
        return np.argmax(output, axis=1) # Trả về chỉ số 0-9

In [2]:
# --- 2. Chuẩn bị dữ liệu ---
# Đọc file (Thay đổi tên file nếu cần: trainMNIST.csv hoặc train.csv)
train_df = pd.read_csv('trainMNIST.csv')
y_raw = train_df['label'].values
X_raw = train_df.drop('label', axis=1).values

# Chuẩn hóa (0-255 -> 0-1)
X = X_raw / 255.0

# One-hot encoding cho y
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y_raw.reshape(-1, 1))

# Chia train/test (80% train, 20% test) để tự kiểm tra
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# --- 3. Thử nghiệm CASE 1: n = 15 lớp ẩn ---
print("\n--- Training với Hidden Layer n = 15 ---")
nn_small = NeuralNetwork(input_size=784, hidden_size=15, output_size=10, learning_rate=0.5)
nn_small.train(X_train, y_train, epochs=500)

# Đánh giá
preds_small = nn_small.predict(X_val)
true_labels = np.argmax(y_val, axis=1)
acc_small = np.mean(preds_small == true_labels)
print(f"Độ chính xác (n=15): {acc_small * 100:.2f}%")

# --- 4. Thử nghiệm CASE 2: n = 100 lớp ẩn ---
print("\n--- Training với Hidden Layer n = 100 ---")
nn_large = NeuralNetwork(input_size=784, hidden_size=100, output_size=10, learning_rate=0.5)
nn_large.train(X_train, y_train, epochs=500)

# Đánh giá
preds_large = nn_large.predict(X_val)
acc_large = np.mean(preds_large == true_labels)
print(f"Độ chính xác (n=100): {acc_large * 100:.2f}%")


--- Training với Hidden Layer n = 15 ---
Epoch 0, Loss: 0.2303
Epoch 50, Loss: 0.2160
Epoch 100, Loss: 0.1409
Epoch 150, Loss: 0.0954
Epoch 200, Loss: 0.0733
Epoch 250, Loss: 0.0610
Epoch 300, Loss: 0.0527
Epoch 350, Loss: 0.0469
Epoch 400, Loss: 0.0426
Epoch 450, Loss: 0.0395
Độ chính xác (n=15): 89.56%

--- Training với Hidden Layer n = 100 ---
Epoch 0, Loss: 0.2304
Epoch 50, Loss: 0.1816
Epoch 100, Loss: 0.0882
Epoch 150, Loss: 0.0599
Epoch 200, Loss: 0.0482
Epoch 250, Loss: 0.0420
Epoch 300, Loss: 0.0381
Epoch 350, Loss: 0.0355
Epoch 400, Loss: 0.0336
Epoch 450, Loss: 0.0321
Độ chính xác (n=100): 90.64%


##  Giải thích kết quả (Dựa trên số liệu thực tế)

Dựa trên kết quả chạy thử nghiệm thực tế:

- **Trường hợp $n=15$**:
  - Loss giảm chậm (Tại Epoch 100 là ~0.14).
  - Độ chính xác: **~89.56%**.
  - _Nhận xét_: Mặc dù cấu trúc rất nhỏ (chỉ 15 nơ-ron ẩn), mạng vẫn hoạt động tốt, chứng tỏ khả năng nén đặc trưng tốt của ANN.

- **Trường hợp $n=100$**:
  - Loss giảm nhanh hơn nhiều (Tại Epoch 100 đã giảm xuống ~0.08, thấp hơn hẳn trường hợp n=15).
  - Độ chính xác: **~90.64%**.
  - _Nhận xét_: Mạng lớn hơn có khả năng học nhanh hơn và nắm bắt các đặc trưng tốt hơn, dẫn đến độ chính xác cao hơn (~1%).

**Kết luận thực nghiệm**:

1.  **Khả năng học (Capacity)**: Tăng số lượng nơ-ron ẩn ($n$) giúp mạng học nhanh hơn (Loss giảm lẹ hơn) và đạt độ chính xác cao hơn.
2.  **Đánh đổi (Trade-off)**: Tăng lượng nơ-ron lên gần 7 lần (15 -> 100) nhưng độ chính xác chỉ tăng thêm khoảng 1%. Trong các bài toán lớn, ta cần cân nhắc việc tăng kích thước mạng quá lớn vì sẽ tốn tài nguyên tính toán mà hiệu quả mang lại có thể bão hòa.